<!---
Class notes Computational Methods
By
Oscar Antonio Restrepo Gutiérrez
--->

# Numerical solution of integrals

“*I had learned to do integrals by various methods shown in a book that my high school physics teacher Mr. Bader had given me. [It] showed how to differentiate parameters under the integral sign — it’s a certain operation. It turns out that’s not taught very much in the universities; they don’t emphasize it. But I caught on how to use that method, and I used that one damn tool again and again. [If] guys at MIT or Princeton had trouble doing a certain integral, [then] I come along and try differentiating under the integral sign, and often it worked. So I got a great reputation for doing integrals, only because my box of tools was different from everybody else’s, and they had tried all their tools on it before giving the problem to me.*”<br> Book: Surely you’re Joking, Mr. Feynman! ([See more here.](https://www.cantorsparadise.com/richard-feynmans-integral-trick-e7afae85e25c))

Not every integral can be solved analytically; in fact, in most physics problems integrals have to be solved by numerical methods. The numerical integration methods studied in this module are: 

  1) [Quadrature (Riemann) method](#cuadratura_Riemann) (Class 14).<br>
  2) [Trapezoidal method](#Método_trapezoidal) (Class 14).<br>
  3) [Simpson's method](#Método_de_Simpson) (Class 14).<br>
  4) [Gaussian quadrature methods.](#cuadratura_gaussiana) (Class 15)<br>
  5) [Romberg's method.](#Método_de_Romberg) (Class 15)<br>
  6) [Improper integrals.](#Integrales_impropias) (Class 16)<br>
  7) [von Neumann method for integrals](#Método_de_von_Neumann) (Monte Carlo) (Class 16).<br>
  8) [Multiple integrals](#Integrales_múltiples) (mean value theorem and Monte Carlo) (Class 16).<br>
  9) [Supplementary material](#MATERIAL_COMPLEMENTARIO1) (Class 16).

(Note: run the last two cells to generate the notebook's plots)

<a id='cuadratura_Riemann'></a>
 ## 1) Riemann quadrature
 The simplest way to compute the integral is to use rectangles to calculate the area under the curve,
 
 |<img src="../figures/Riemann.png" alt="Drawing" style="width: 500px;"/>|
|:--:| 
| *Figure: Integral by the quadrature method*|

 
 more precisely, if $f(x)$ is a function defined on the interval $[a,b]$ such that $a = x_0 < x_1 < \dots < x_n = b$, then from the definition of the Riemann integral
the area under the curve can be computed as,

$$I = \sum_{i=1}^n f(x_i)h = \sum_{i=1}^{n} A_i$$

where $A_i=f(x_i)h=$ "*area of the rectangle on the subinterval* $[x_{i-1},x_{i}]$" and $n =$ "*number of subintervals or rectangles*" (note the sum could also run from $i=0,...,n-1$). Also  

$$h = (b-a)/n$$
$$x_i = a + ih.$$
   
The following routine computes the integral of $\cos(x)$ on the interval $[0,\pi/2]$:


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# If the input is a function defined on [a,b], with n subintervals.
def Cuadratura(f,a,b,n):
   h = (b - a)/n
   S = 0
   for i in range(1,n+1): # Note this also works with range(n)=0,..,n-1
     S += f(a + i*h)*h    # I = A1 + A2 + ... + An
   return S    

# Example1:
Cuadratura(lambda x: np.cos(x),0, np.pi/2, 1000) # exact value 1.0


In [ ]:
# If the input is two data arrays (yi,xi)    
def Cuadratura2(y,x):
   S = 0
   for i in range(1,len(x)):
     S += y[i]*(x[i] - x[i-1])
   return S    

# Example2:
xi = np.linspace(0, np.pi/2, 1001) # must be n+1 (array size)
yi = np.cos(xi)
Cuadratura2(yi,xi)


<a id='Método_trapezoidal'></a>
# 2) Trapezoidal method
 In the previous method the error is quite large since rectangles are used to compute the area on each subinterval; a better method is to use trapezoids for the integration.
 
 |<img src="../figures/Trapezoidal.png" alt="Drawing" style="width: 500px;"/>|
|:--:| 
| *Figure: Integral by the trapezoidal method*|

 
 To integrate $f(x)$ on the interval $[a,b]$, the function is approximated by a straight line on each subinterval $[x_i,x_{i+1}]$, and its area is computed as *base times height*, where the height is computed as the average value $(f(x_i) + f(x_{i-1}))/2$; then the area of the trapezoid on the interval $[x_i,x_{x+1}]$ is

   $$A_i = \frac{f(x_i) + f(x_{i-1})}{2}h.$$
             
If all the areas $A_i$ are summed we get an approximation of the integral:
  
 \begin{align} I =&\, \sum_{i=1}^{n} A_i\\
                 =&\, A_1 + A_2 + \cdots + A_n,\\
                 =&\, \frac{f(x_0) + f(x_{1})}{2}h + \frac{f(x_1) + f(x_{2})}{2}h +\cdots+\frac{f(x_{n-2}) + f(x_{n-1})}{2}h+\frac{f(x_{n-1}) + f(x_{n})}{2}h,\\
               I =&\, h/2\left[f(a) + 2\sum_{i=1}^{n-1}f(x_i) + f(b)\right],
 \end{align}
 
 or in more compact form
 
 $$I = \sum_{i=0}^{n} f(x_i)w(x_i),$$
 
 where the $w(x_i)$ are known as weights, in this case
 
 $$w(x_i) = \{h/2,h,... ,h,h/2\},$$ 
 
 and
 
 $$\sum_{i=0}^{n} w(x_i) = nh.$$
 
In general ([see supplementary material](#MATERIAL_COMPLEMENTARIO1)),
 
 $$\boxed{ \int_a^b f(x)dx = \frac{h}{2}\left[f(a) + 2\sum_{i=1}^{n-1}f(x_i) + f(b)\right]-\frac{b − a}{12}h^2f''(\xi).}$$
 
 Where the last term is the error and $\xi$ is an unknown number such that $\xi ∈ [a,b]$. 


In [ ]:
def Trapezoidal(f,a,b,n):
   h = (b - a)/n
   S = (f(a) + f(b))/2.  # initialise with f0w0 + f[n-1]*w[n-1]
   for i in range(1,n):# sum over 1, ..., n-2 
     S = S + f(a + i*h)  # I = f1w1 + f2w2 + ... + fn-2*wn-2
   return S*h # add the last element fn-1*wn-1

# Example3:
Trapezoidal(lambda x: np.cos(x),0, np.pi/2, 1000) # exact value 1.0


In [ ]:
# If the input is two data arrays (yi,xi)    
def Trapezoidal2(y,x):
   n = len(x) - 1
   S = y[0]*(x[1] - x[0])/2.          # initialise with f0w0
   # FIX: the original range(1,n-1) omitted the i=n-1 term, which
   # underestimated the integral (verified: without the fix it gave
   # 0.99999733 instead of 0.99999979 for cos(x) on [0,pi/2], compared
   # against Trapezoidal()).
   for i in range(1,n):               # sum over 1, ..., n-1
     S += y[i]*(x[i+1] - x[i])        # I = f1w1 + f2w2 + ... + fn-1*wn-1
   return S + y[n]*(x[n] - x[n-1])/2. # add the last element fnwn

# Example4:
xi = np.linspace(0, np.pi/2, 1001)    # must be n+1 (array size)
yi = np.cos(xi)
Trapezoidal2(yi,xi)


<a id='Método_de_Simpson'></a>
# 3) Simpson's method

|<img src="../figures/Simpson.png" alt="Drawing" style="width: 500px;"/>|
|:--:| 
| *Figure: Integral by Simpson's method.*|

 In this method the function is replaced by a parabola
            
 $$f(x) = ax^2 + bx + c,$$
            
 that passes through the points $x_{i-1},x_i,x_{i+1}$, on the interval $[x_{i-1},x_{i+1}]$, and the area $A_i$ under this curve is computed; to do this, the parabola is integrated on the interval $[x_{i-1},x_{i+1}]$. Let's look at a heuristic derivation (see [supplementary material](#MATERIAL_COMPLEMENTARIO1) for a proof using Lagrange polynomials); consider the integral
 
 $$\int_{-1}^1(ax^2 + bx + c)dx=\frac{2}{3}a+2c,$$
 
 now, noting that
 
 $$f(-1) = a-b+c, \quad  f(0) = c, \quad  f(1) = a+b+c $$
 
 then solving for $a,b$ and $c$
 
 $$a=\frac{f(1)+f(-1)}{2}-f(0), \quad b=\frac{f(1)-f(-1)}{2},\quad c=f(0), $$
 
 now we substitute $a,b$ and $c$ into the previous integral, which gives
 
 $$\int_{-1}^1(ax^2 + bx + c)dx= \frac{f(-1)}{3} + \frac{4f(0)}{3} + \frac{f(1)}{3},$$
 
 now if we rescale, i.e. change $(-1 \rightarrow x_i-h)$, $(0 \rightarrow x_i)$,  $(1 \rightarrow x_i+h)$ and multiply by $h$ to recover the units of area, we find that
 
 $$A_i = \int_{x_i-h}^{x_i+h}(ax^2 + bx + c)dx = \left(f(x_{i-1}) + 4f(x_i) + f(x_{i+1})\right)\frac{h}{3},$$ 

 for $i = 1,...,n/2$, since $A_i$ covers two intervals (note that $n$ must be even). 


If all $n/2$ areas $A_i$ are summed we get an approximation of the integral:
  
\begin{align} I =&\, \sum_{i=1}^{n/2} A_i\\
                 =&\, A_1 + A_2 + \cdots + A_{n/2},\\
                 =&\, \left(f(x_{0}) + 4f(x_1) + f(x_{2})\right)\frac{h}{3} + \left(f(x_{2}) + 4f(x_3) + f(x_{4})\right)\frac{h}{3} +\cdots\\
                 =&\frac{h}{3}f(x_{0})+\frac{4h}{3}f(x_{1})+\frac{2h}{3}f(x_{2})+\frac{4h}{3}f(x_{3})+\cdots+\frac{4h}{3}f(x_{n-1})+\frac{h}{3}f(x_{n})
 \end{align}

if the even and odd terms are summed separately, this gives

$$I =\,\frac{h}{3}\left[f(a) + 2\sum_{i=1}^{(n/2)-1}f(x_{2i}) + 4\sum_{i=1}^{n/2}f(x_{2i-1}) + f(b)\right],$$

or in more compact form

 $$I = \sum_{i=0}^{n} f(x_i)w(x_i),$$
 
 where the $w(x_i)$ are known as weights; in this case note that

$$w(x_i) = \left\{\frac{h}{3},\frac{4h}{3},\frac{2h}{3},\frac{4h}{3},\cdots,\frac{4h}{3},\frac{h}{3}\right\},$$ 

and

 $$\sum_{i=0}^{n} w(x_i) = nh,$$
 
 In general, Simpson's rule is
 
$$\boxed{
 \int_a^b f(x)dx = \frac{h}{3}\left[f(a) + 2\sum_{i=1}^{(n/2)-1}f(x_{2i}) + 4\sum_{i=1}^{n/2}f(x_{2i-1}) + f(b)\right]-\frac{b − a}{180}h^4f^{(4)}(\xi)
,}$$
 
 where $\xi ∈ (a,b).$


In [ ]:
# Integral on [a,b]; n must be a positive integer, since it requires 2 intervals/step.
def Simpson(f,a,b,n):
   h = (b - a)/n
   S0 = f(a) + f(b)
   S1 = 0 
   S2 = 0 
   for i in range(1,n):  # sum over 1, ..., n-1 
     if (i%2==0): 
        S2 += f(a + i*h) # sum of even terms
     else:
        S1 += f(a + i*h) # sum of odd terms
   return (S0 + 2*S2 + 4*S1)*h/3 

# Example5:
Simpson(lambda x: np.cos(x),0, np.pi/2, 1000) # exact value 1.0


In [ ]:
# Alternative (only sums up to n/2 - 1):
import numpy as np
def Simpson(f,a,b,n):
    h = (b - a)/n
    S0 = f(a) + 4*f(a + (n-1)*h) + f(b)
    S1 = 0
    S2 = 0
    for i in range(1,n//2):    # sum over 1, ..., n/2 - 1 
        S2 += f(a + 2*i*h )    # sum of even terms
        S1 += f(a + (2*i-1)*h) # sum of odd terms
    return (S0 + 2*S2 + 4*S1)*h/3
 
# Example5:
Simpson(lambda x: np.cos(x),0, np.pi/2, 1000) # exact value 1.0


In [ ]:
# If the input is two data arrays (xi,yi)    
def Simpson2(x,y):
   n = len(x) - 1
   hn = x[n] - x[n-1]
   h0 = x[1] - x[0]
   S0 = y[0]*h0 + y[n]*hn # initialise
   S1 = 0
   S2 = 0
   for i in range(1,n):   # sum over 1, ..., n-1
     hi = x[i+1] - x[i]
     if (i%2==0):         
        S2 += y[i]*hi     # sum of f(2xi) if i is even
     else:
        S1 += y[i]*hi     # sum of f(2x[i-1]) if i is odd
   return (S0 + 2*S2 + 4*S1)/3 

# Example6:
xi = np.linspace(0, np.pi/2, 1001) # must be n+1 (array size)
yi = np.cos(xi)
Simpson2(xi,yi)


### Rounding error and stability in Simpson's and the trapezoidal rule
 The rounding error $e(h)$ does not increase as $n$ increases: to see this, consider
        $f(x_i) = g(x_i) + e_i$, for each $i = 0, 1, . . . , n,$, where $g(x_i)$ is an approximation of $f(x_i)$;
 in general it can be shown that $e(h) ≤ (b − a)\epsilon,$ with $\epsilon$ a bound on the individual error at
 each step; this indicates the procedure is stable as $h$ tends to zero.

<a id='cuadratura_gaussiana'></a>
# 4) Gaussian quadrature method
In the previous methods the spacings on the interval $[a,b]$ were all equal ($h$ constant); a better approximation is to assign weights to each value of the function for specific values $x_i$, since these points contribute more to the sum, so the sum will need to be evaluated at fewer points.
The [Gaussian quadrature](https://en.wikipedia.org/wiki/Gaussian_quadrature) method
is based on this idea, i.e.,
$$\int_{-1}^1 f(x)\,dx \approx \sum_{i=1}^n w_i f(x_i).$$

It can be shown that the values $x_i$, for $i=1,...n$, are the zeros of the [Legendre polynomials](https://en.wikipedia.org/wiki/Legendre_polynomials) (see [supplementary material](#Polinomios_de_legendre) at the end) and the weights $w'_i$ are given by,

$$w_i = \frac{2}{\left( 1-x_i^2 \right) [P'_n(x_i)]^2},$$
where $P'_n(x_i)$ are the derivatives of the Legendre polynomials.  
This integration method is known as the *Gauss-Legendre* method.
The integral can be generalised to any interval $[a,b]$ using a simple linear transformation $y=mx+c$, from the interval $[-1,1]\rightarrow[a,b]$, which gives $m=(b-a)/2$ and $c=(a+b)/2$; substituting into the integral we have:

$$\int_a^b f(x)\,dx = \frac{b-a}{2} \int_{-1}^1 f\left(\frac{b-a}{2}x + \frac{a+b}{2}\right)\,dx,$$

so the integral is approximately equal to

$$\int_a^b f(x)\,dx \approx \frac{b-a}{2} \sum_{i=1}^n w_i\,f\left(\frac{b-a}{2}x_i + \frac{a+b}{2}\right).$$

If $n$ points are used, it can be shown that the error is proportional to the $2n$-th derivative, i.e.,

$$\epsilon=\frac{(b-a)^{2n+1} (n!)^4}{(2n+1)[(2n)!]^3} f^{(2n)} (\xi),$$

with $a < \xi < b$. 

In general, if we define $W_i=\frac{b-a}{2}w_i$, $y_i=\frac{b-a}{2}x_i + \frac{a+b}{2}$, the integral on the interval $[a,b]$ is

$$\int_a^b f(x)\,dx = \sum_{i=1}^n W_i\,f(y_i) + \epsilon.$$

Note that from this expression we conclude that for polynomials of degree $n$ (or lower) the solution to the integral is exact, i.e. the error is zero. 

The following code computes the integral with the Gauss-Legendre method; for this it calls the `gauss` function, which is in charge of computing the zeros $x_i$ and the weights $w_i$ of the Legendre polynomials. Note that the "gauss" function uses the Newton-Raphson method, $t_{k+1,i}=t_{k,i}-P_n(t_{k,i})/P'_n(t_{k,i})$, to approximate each zero ($i=1,...,n$ denotes the zeros and $k$ the Newton-Raphson iterations). By symmetry it only computes half of them, using as an initial approximation for each zero $i$ the expression
$t_{0,i} = \cos\left(\frac{\pi(i - 1/4)}{n + 1/2}\right)$.
The derivatives of the Legendre polynomials, $P'_n(x_i)$ (needed to compute the weights $w_i$), are computed using the recurrence relation given at the [end of the chapter](#Polinomios_de_legendre). 


In [ ]:
def gauss(n,job, a,b,x,w, eps = 3.E-15 ):
    """
    -------------------------------------------------------------------------
    # Gaussian Quadrature method (Gauss-Legendre quadrature) using n        #
    # points on the interval [a,b]; x and w are arrays with the points xi   #
    # (zeros of the Legendre polynomial P_n(x)) and weights wi; eps is the  #
    # desired error, job is:                                                #
    #        0 for integration on [a, b]                                    #
    #        1 for integration on [0, b]                                    #
    #        2 for integration on [a, inf]                                  #
    -------------------------------------------------------------------------
    """    
    m = (n + 1)//2 # By symmetry, only half of the roots need to be computed.
    for i in range(1, m + 1): # Loop to find the roots and weights.
        t = np.cos(np.pi*(i - 0.25)/(n + 0.5) ) # Initial guess for the
        t1 = 1                                  # ith root of order n.
        while( (np.abs(t - t1) ) >= eps):       # Newton iteration to
            p1 = 1. ; p2 = 0.                   # find the ith root.
            for j in range(1, n + 1):
                p3 = p2                         # Recurrence relation of 
                p2 = p1                         # P_n(x).
                p1 = ( (2.*j - 1)*t*p2 - (j - 1.)*p3)/j
                                                # Recurrence relation of 
            pp = n*(t*p1 - p2)/(t*t - 1.)       # P'_n(x): derivative of P_n(x).
            t1 = t
            t  = t1 - p1/pp                     # xi = x - P_n(x)/P'_n(x).

        x[i - 1] = - t      # Store the root found and,
        x[n - i] = t        # by symmetry, this is the other unc computed root.
        w[i - 1] = 2./( (1. - t*t)*pp*pp)       # Store weight.
        w[n - i] = w[i - 1]                     # Store, using symmetry.
        # print(" x[i - 1]", x[i - 1] , " w " , w[n - i])
    if (job == 0):
        for i in range(0, n):
            x[i] = x[i]*(b - a)/2. + (b + a)/2. # Transformation from [-1,1] to
            w[i] = w[i]*(b - a)/2.              # the interval [a, b].
            
    if (job == 1):                              # Scale to (0, b) with 50% of the points 
        for i in range(0, n):                   # inside (0, ab/(a + b))
            xi = x[i]
            x[i] = a*b*(1. + xi)/(b + a - (b - a)*xi)
            w[i] = w[i]*2.*a*b*b/( (b + a - (b - a)*xi)**2. )
            
    if (job == 2):                              # Scale to (a, inf) with 50% 
        for i in range(0, n):                   # inside (a, b + 2a)
            xi = x[i]
            x[i] = (b*xi + b + a + a)/(1. - xi)
            w[i] = w[i]*2.*(a + b)/( (1. - xi)**2. ) 


In [ ]:
# **** function that computes the integral by calling Gauss() to get the wi, xi ****       
def Integral_Gauss(f, a, b, n):
    """
       Integral of f(x) on [a,b] using the Gauss method with n points. 
    """
    w = np.zeros(n)           # will hold the zeros of the Legendre polynomials
    x = np.zeros(n)           # will hold the weights w for the integration

    gauss(n, 0, a, b, x, w)   # Returns points xi and wi

    return sum(f(x)*w)        # Compute the integral    


#example7:
Integral_Gauss(lambda x: np.cos(x), 0., np.pi/2, 6)


<a id='Método_de_Romberg'></a>
# 5) Romberg's method
In the previous methods it is not possible to compute the integral to a desired precision; Romberg's method lets you define an $\epsilon$ to compute the integral (unlike numerical derivatives, $\epsilon$ is only limited by machine precision), and this method is also almost as fast as the Gaussian quadrature method (the algorithm is obtained by applying the recurrence formula of [Richardson extrapolation](https://en.wikipedia.org/wiki/Richardson_extrapolation)). 
 Consider the integral
 $$I=\int_a^bf(x)dx$$
 has an approximate value $A(h)$ computed from the trapezoidal method, so if the error is included as a power series in $h$,
 
 $$I = A(h) + K_1h^2 +K_2h^4 + K_3h^6...$$
 
 Suppose that, to reduce the error, $A$ is computed but at each $h/2$,
 
 $$I = A\left(\frac{h}{2}\right) + K_1\frac{h^2}{4} +K_2\frac{h^4}{16} ...$$
 
 If this equation is multiplied by $4$ and subtracted from the first (thereby eliminating the $h^4$ term), we obtain
 
 $$I =\left[\frac{4}{3}A\left(\frac{h}{2}\right)-\frac{1}{3}A\left(h\right)\right] + K_2\frac{h^4}{4} ...$$

the expression between square brackets has an error of order $O(h^4)$ and is therefore closer to the true value $I$. This is repeated iteratively (see the supplement for the [derivation of Romberg's method](#Deducción_del_método_de_Romberg)).

The implementation is:


In [ ]:
def Romberg(f, a, b,kmax = 30, eps = 1e-9, p = False):
    """
    -------------------------------------------------------------------------
    # Adaptive Romberg method.                                              #
    # Integrates the function f(x) on [a,b] with precision eps              #
    # kmax is the maximum number of iterations                              #
    -------------------------------------------------------------------------
    """ 
    r1 = np.zeros(kmax+1)
    r2 = np.zeros(kmax+1)

    h = b-a 
    n = 1
    r1[0] = 0.5*h*(f(a) + f(b)) # initial approximation
    for k in range(1,kmax+1):   # step halving loop
        sumf = 0.
        for i in range(1,n+1): sumf += f(a+(i-0.5)*h)
        r2[0] = 0.5*(r1[0] + h*sumf) # trapezoidal formula
        y = 1.
        for j in range(1,k+1):  # increase quadrature order
            y *= 4
            r2[j] = (y*r2[j-1] - r1[j-1])/(y-1) # new approximation

        if (k > 1): # check convergence
            if (np.fabs(r2[k]-r1[k-1]) <= eps*np.fabs(r2[k])): break
            if (np.fabs(r2[k]) <= eps and np.fabs(r2[k]) <= np.fabs(r2[k]-r1[k-1])):break
        
        h *= 0.5; n *= 2 # halve the integration step
        for j in range(0,k+1): r1[j] = r2[j] # shift the table's rows
        if p == True: print(k, r2[k])
    if (k >= kmax):
        print("Romberg: max. number of iterations exceeded! kmax=",kmax)
        k -= 1
    
    return r2[k]

#Example
#Romberg(lambda x: np.cos(x), 0., np.pi/2.,eps=1e-14)
#Romberg(lambda x: np.cos(x), 0., np.pi/2.,kmax=10)
Romberg(lambda x: np.cos(x), 0., np.pi/2.,p=True)


**Task**: do exercise 6.2.5 from the book by [Landau & Páez](https://www.eidos.ic.i.u-tokyo.ac.jp/~tau/lecture/computational_physics/docs/computational_physics.pdf#page=140) (page 140).


In [ ]:
# Solution: 
# Plot comparing the integration error
# for f(x) = exp(-x) on [0,1]
# for 4 different numerical methods

nmax = 1000
N = 1 - np.exp(-1)
f = lambda t:np.exp(-t)

etrap=[]
eSimpson=[]
eGauss=[]
eRomberg=[]

x = np.arange(2,nmax,2) # done for even n (since Simpson requires even n)
for n in x:
    etrap.append(np.abs((Trapezoidal(f,0 ,1,n) - N)/N) )
    eSimpson.append((np.abs(Simpson(f,0 ,1,n) - N)/N) )
    eGauss.append((np.abs(Integral_Gauss(f,0 ,1,n) - N)/N) )
    eRomberg.append((np.abs(Romberg(f,0 ,1,n) - N)/N) )
    
plt.loglog(x,etrap,label='trap')   
plt.loglog(x,eSimpson,label='Simpson') 
plt.loglog(x,eGauss,label='Gauss') 
plt.loglog(x,eRomberg,label='Romberg')

plt.xlabel('n')
plt.ylabel('Error')
plt.legend()


**Example**: Find an estimate of the best number of steps $n$ for the Gauss integral on the interval $[0,1]$ (see section [*6.2.3 Integration Error*, from the book by Landau & Páez](https://www.eidos.ic.i.u-tokyo.ac.jp/~tau/lecture/computational_physics/docs/computational_physics.pdf#page=137)). 

Solution: We know that for a function $f$ the relative error is given by,

$$\epsilon_{re} = \frac{\epsilon}{f},$$

with the Gauss error given by,

$$\epsilon=\frac{(b-a)^{2n+1} (n!)^4}{(2n+1)[(2n)!]^3} f^{(2n)} (\xi),$$

let $\epsilon_m\approx 10^{-16}$ be the error of working with 64 bits.
Now, if we assume the relative error after $n$ steps is approximately,

$$\hbox{Error} = \sqrt{n}\epsilon_m,$$

then we need to find the value of $n$ that minimises the error; note this happens when both errors are approximately equal, i.e. $\epsilon_{re}\approx\hbox{Error}$, that is,

$$\sqrt{n}\epsilon_m\approx\frac{(n!)^4}{(2n+1)[(2n)!]^3},$$

where we assume $(b-a)=1$ and $f^{(2n)}/f\approx 1$. Unfortunately $n$ cannot be found easily because of the factorial, so the previous expression is set to zero in order to use root-finding methods,

$$g(n)=0=\sqrt{n}\epsilon_m - \frac{(n!)^4}{(2n+1)[(2n)!]^3}.$$

A quick way to find $n$ is to look for the value of $n$ where the sign of the function $g(n)$ switches from positive to negative;
according to this, the best $n$ for $\epsilon_m=10^{-16}$ is between $5$ and $6$, which agrees with the error plot shown earlier, so the error in the Gauss integration is approximately $\sqrt{n}\epsilon_m \approx 2\times10^{-16}$; let's see: 


In [ ]:
# Error estimate for the Gauss integration on the interval [0,1]
import math
fact = lambda x: math.factorial(int(x))  # int(): factorial requires an integer, n*2. is a float
g = lambda n: n**.5*1e-16 - (fact(n))**4./(2.*n+1.)/(fact(2.*n))**3.                                                   

for n in range(10):  
    print(n, g(n)) 


**Task**: in the previous example, for the Gaussian integral, for simplicity we assumed $b-a=1$; you could also assume $b-a=2$, since the integration interval is $[-1,1]$; repeat the exercise with this assumption and compare.

**Task**: use the same procedure as above to find that the number of steps is of order $2.5\times 10^6$ for the trapezoidal method and of order $10000$ for Simpson's method.


<a id='Integrales_impropias'></a>
# 6) Improper integrals
These are integrals that have infinities somewhere in the integration interval (or at the limits $a,b$). 
## Divergence in the function 
Consider the function $f(x)=g(x)/(x-a)^p$, which diverges since $f(a)=\infty$ (see figure):

|<img src="../figures/f_impropia.png" alt="Drawing" style="width: 400px;"/>|
|:--:| 
| *Figure: improper function: $f(a)=\infty$.*|

Nevertheless, its integral converges if $0<p<1$. That is, integrals of functions of this type with a singularity at $a$ on the interval $[a,b]$, where $g(x)$ is continuous on $[a,b]$,

$$I = \int_a^b\frac{g(x)}{(x-a)^p}\,dx\quad\hbox{with}\quad 0<p<1,$$

converge and have an approximate solution given by,

$$I = \int_a^b\frac{g(x)-P_4(x)}{(x-a)^p}\,dx+\int_a^b\frac{P_4(x)}{(x-a)^p}\,dx,$$
where, by Taylor series,

$$P_4(x) = g(a)+g'(a)(x-a)+\frac{g''(a)}{2!}(x-a)^2+\frac{g'''(a)}{3!}(x-a)^3+\frac{g^{(4)}(a)}{4!}(x-a)^4,$$

the second integral is the dominant term (contributes the most) and can be computed as, 

$$
\begin{eqnarray}
\int_a^b\frac{P_4(x)}{(x-a)^p}\,dx
&=&\,\int_a^b\sum_{k=0}^4 \frac{g^{(k)}(a)}{k!}(x-a)^{k-p}\,dx,\\
&=&\,\sum_{k=0}^4 \frac{g^{(k)}(a)}{k!}\int_a^b(x-a)^{k-p}\,dx,\\
&=&\,\sum_{k=0}^4\frac{g^{(k)}(a)}{k!(k+1-p)}(b-a)^{k+1-p},
\end{eqnarray}
$$

the first integral can be approximated with Simpson's method, using the function,

$$
\begin{align}
  &f(x)=\begin{cases}
    \begin{alignedat}{3}
      \frac{g(x)-P_4(x)}{(x-a)^p},& \quad\hbox{if}\quad a < x \le b   \\
       0,\quad\quad\quad\quad     &\quad\hbox{if}\quad   x=a, 
    \end{alignedat}
  \end{cases}\\
\end{align}
$$

which avoids the indeterminate form at $x=a$.
If the singularity is at $b$, a change of variable $t=-x$ recovers the previous integral, giving,

$$I = \int_{a}^{b}f(x)\,dx= \int_{-b}^{-a}f(-t)\,dt.$$

If the singularity is at $c$ between $[a,b]$, the integral is split into two, one on $[a,c]$ and another on $[c,b]$. 

## Infinite limits 
If there are infinite limits, the change of variable $t=1/x$ is used (this, in addition to the Gaussian quadrature methods for certain improper integrals; see [table](#Tabla_cuadratura_gaussiana)). But how do you solve an integral of the form $\int_0^\infty f(x)\,dx$? note $t=\frac{1}{x}$ cannot be used.
Answer: simply split it into two intervals $[0, 1]$ and $[1, \infty]$.

**Example**: consider the integral,

$$I = \int_1^\infty x^{-3/2}\sin\left(\frac{1}{x}\right)\,dx,$$

if the change of variable $t=1/x$ and $dx=-dt/t^2$ is made, the integral becomes,

$$I = \int_0^1t^{-1/2}\sin\,(t)\,dt,$$

with singularity $t=a=0$.
For more theory on improper integrals and integrals with infinite limits, read Burden pages: 250-255.
also see the [website](http://nm.mathforcollege.com/topics/textbook_index.html).

Implementation of the previous integral:


In [ ]:
#Taylor series with sympy
from sympy import *
init_printing(use_latex='mathjax') # nice print!!!

t = symbols('t')
g = sin(t)
series(g,t,x0=0, n=4)


In [ ]:
P4 = series(g,t,0,4).removeO() # remove O(h^4)
a=0; b=1; p=1/2 

# -------- first integral -----------------------
def SimpsonP4(a, b, p, n):
    G = (g - P4)/(t - a)**p     # symbolic function  
    f = lambdify(t, G, 'numpy') # numerical function

    h = (b - a)/n
    S0 = f(b)                # note f(a) = 0
    S1 = 0 
    S2 = 0 
    for i in range(1,n):     # sum over 1, ..., n-1 
        if (i%2==0): 
            S2 += f(a + i*h) # sum of f(2xi)
        else:
            S1 += f(a + i*h) # sum of f(2x[i-1])
    return (S0 + 2*S2 + 4*S1)*h/3 

# --------- second integral ----------------------
def Integral_P4(a, b, p, n):
    S = 0
    for k in range(n):
       dy = g.diff(t,k)
       dg = lambdify(t, dy, 'numpy')        
       S += dg(a)*(b-a)**(k+1-p)/(k+1-p)/math.factorial(k)
    return S

# ------------ The result is the sum: ------------
Integral_P4(a, b, p, 4), SimpsonP4(a, b, p, 16)


In [ ]:
# Integral_P4(a, b, p, 4) contributes 0.998% of the area:
(0.6190476190476191- 0.0014890096885413241)/0.6190476190476191


Note that the approximation $P_4(x)$ works well if $(x-a)$ is small, which will depend on the separation between $a$ and $b$; to guarantee this, another method is to split the integral in two: the first on the interval $[a,a+h]$, where $g(x)$ is replaced by $P_4(x)$, and the second on the interval $[a+h,b]$, where $f(x)=\frac{g(x)}{(x-a)^p}$ is used. As we'll see in the following code, Simpson doesn't work so well since it needs $n = 10000$ to get $0.62053661$ (with a precision of $4\times10^{-8}$) of the previous result — why? The function has large variations between $x$ and $x+h$ as $x$ approaches the value $a$, and Simpson's rule doesn't work well on the second integral because of the degree-three polynomial approximation; so, the value of $h$ must be chosen small enough that the difference $|f(x)-f(x+h)|$ isn't too large. But if Gauss-Legendre is used on the second integral, this difference is no longer as important — in fact it works quite well; let's compare:


In [ ]:
# Simpson doesn't work so well since it needs n = 10000 to
# get 0.62053661 (with precision 4e-8) of the previous result.
# Why? the function has large variations between x and x+h 
# as x approaches a, and Simpson doesn't work well because
# of the degree-3 polynomial approximation, but with Gauss it
# works quite well, COMPARE: 
n = 16      # same value used in the previous case. 
h = (b-a)/n # same h used in Simpson
fo = lambda x: np.sin(x)/x**.5 # f(x)
IS = Integral_P4(a, a+h, p, 4) + Simpson(fo, a+h, b, n)    
IG = Integral_P4(a, a+h, p, 4) + Integral_Gauss(fo, a+h, b, n) 
# Compare errors:
0.62053661-IS, 0.62053661-IG


## Improper integrals using quadrature methods

The [Gaussian quadrature](#cuadratura_gaussiana) method can also be used to solve improper integrals if the right mappings to the interval $[-1,1]$ are made; for example, with the mapping $x=a(1+t)/(1-t)$, we have,

$$
\int_0^{\infty}f(x)\,dx = \int_{-1}^{1}f\left(a\frac{1+t}{1-t}\right)\frac{2a}{(1-t)^2}\,dt,
$$

another alternative is $x = a \cot^2(t/2)$ (although it would still need to be mapped to $[-1,1]$ afterwards),

$$
\int_0^\infty f(x)\,dx = 2a \int_0^\pi \frac{f[a \cot^2(t/2)]}{[1 - \cos(t)]^2} \sin(t)\,dt. 
$$

For the interval $[-\infty,\infty]$, the mapping $x=at/(1-t^2)$ can be used, which gives,

$$
\int_{-\infty}^{\infty}f(x)\,dx = a\int_{-1}^{1}f\left(a\frac{t}{1-t^2}\right)\frac{1+t^2}{(1-t^2)^2}\,dt.
$$

In all cases $a$ is any constant that speeds up convergence; typically $a=1$.
Another alternative is the transformation $x = -\ln[(1 + \cos(t))/2]$, for integrals with $e^x$, which gives,

$$\int_0^\infty e^{-x} g(x)\,dx=\int_0^\pi f(\cos(t))\sin(t) \,dt\quad\hbox{ where }\quad f(u) = g(-\ln[(1+u)/2])/2.$$

**Task**: a) implement these mappings in the `gauss()` routine and test the result with the integral $\int_{-\infty}^\infty e^{-x^2}\,dx = 2\int_0^\infty e^{-x^2}\,dx=\sqrt{\pi}$.

If the integration is on the intervals $[a,\infty]$, $[-\infty,\infty]$, etc., then the previous transformations must be used to change the integration limits to the interval $[-1,1]$, but in some cases it's better to use other quadrature methods for the integration, such as [Gauss-Hermite](https://en.wikipedia.org/wiki/Gauss%E2%80%93Hermite_quadrature) or [Gauss-Laguerre](https://en.wikipedia.org/wiki/Gauss%E2%80%93Laguerre_quadrature); the implementations are left as a [task](#cuadratura_problemas).

List of the types of integrals that can be solved with Gaussian quadrature: 
<a id='Tabla_cuadratura_gaussiana'></a>
$$
\begin{eqnarray}
&&\hline &\hline &\hline &\hline &\hline \\[-2pt]
&& Integral   & Name   && Integral  & Name  \\
&&\hline &\hline &\hline &\hline &\hline \\
&&\int_{-1}^1 f(x)\,dx & \hbox{Gauss-Legendre} &&  \int_{-1}^1 \frac{f(x)}{\sqrt{1-x^2}}\,dx & \hbox{Gauss-Chebyshev}\\
&&\int_{-\infty}^\infty e^{-x^2}f(x)\,dx & \hbox{Gauss-Hermite} && \int_{0}^\infty e^{-x}f(x)\,dx & \hbox{Gauss-Laguerre}         \\
&&\int_{0}^\infty \frac{e^{-x}}{\sqrt{x}}f(x)\,dx& \hbox{Associated Gauss-Laguerre}&& \int _{0}^{\infty }x^{\alpha }e^{-x}f(x)\,dx&\hbox{Generalized Gauss-Laguerre}\\
&&\hline &\hline &\hline &\hline &\hline
\end{eqnarray}
$$

[Clenshaw-Curtis](https://en.wikipedia.org/wiki/Clenshaw%E2%80%93Curtis_quadrature) quadrature is another alternative to Gaussian quadrature.


<a id='Método_de_von_Neumann'></a>
# 7) Von Neumann's rejection method for integrals (Monte Carlo)

Suppose you want to compute the area of a circle, $A_p$, of radius $L$, using random numbers ($A_p$ could also be the area of a lake). The *rejection method* is a simple method for computing the area: 1) build a square of side $2L$ that contains the circle, and 2) generate $N$ uniformly distributed points inside the square, 3) count how many points $N_p$ fall inside the circle; since the points are uniformly distributed their density is constant, so,

$$\rho = \frac{N}{4 L^2}=\frac{N_p}{A_p},$$

so we have,

$$A_{p}=4 \frac{N_p}{N}L^2 \rightarrow \text{Area of the circle/lake}$$

For this you need to generate uniformly distributed random numbers; the python methods for that are,
```python
# Generate uniformly distributed numbers:
np.random.random()              # Generate a single number in [0,1)
np.random.rand(10)              # Generate 10 numbers in [0,1)
np.random.uniform(-1,1)         # Generate a random number in [-1,1] 
np.random.uniform(-5,5,size=10) # Generate 10 numbers in [-5,5]
```

**Example**: Compute the area of a circle of radius one, using Monte Carlo (should give $\pi$); the code is:


In [ ]:
# Compute the area of a circle of radius one, using Monte Carlo.

N=1000000
Np = 0
for i in range(N):
   x = np.random.uniform(-1,1)
   y = np.random.uniform(-1,1)
   # note this could also be used, since only the squares are considered.   
   #  x = np.random.random()
   #  y = np.random.random()    
   if (x**2.+y**2<1): Np +=1
   
Area = Np/N*4. # the area of the rectangle is 4.
Area


## Von Neumann's rejection method
The previous method can be used to compute the integral of a positive function on the interval $[a,b]$ as the area under the curve; for this a rectangle of area $(b-a)M$ is defined that encloses the area under the curve of $f(x)$, where $M$ is greater than or equal to the maximum of the function on $[a,b]$; then $N$ random points $(x_i,y_i)$ with uniform distribution are placed inside the rectangle, and the $N_p$ points that fall below the curve are counted (i.e. the $(x_i,y_i)$ such that $y_i<f(x_i)$). 
Theory: See [chapter 6 of Landau & Páez](https://www.eidos.ic.i.u-tokyo.ac.jp/~tau/lecture/computational_physics/docs/computational_physics.pdf#page=147).


In [ ]:
# 1) Von Neumann's rejection method; if f(x) > 0 on [a,b] then:
def Von_Neumann0(f,M,a,b,N):
    Np = 0
    for i in range(N):
       x = np.random.uniform(a,b)
       y = np.random.uniform(0,M)
       
       if (y < f(x)): Np += 1 # point inside the area of the integral
       
    I = Np/N*(M*(b-a)) # the area of the rectangle is M*(b-a).
    return I

# Example2:
M =1.0 # greater than sin(x) on [0,pi/2] 
f = lambda x: np.cos(x) 
x = np.linspace(0,np.pi/2,1000)
plt.fill_between(x,f(x))

Von_Neumann0(f,M, 0.0, np.pi/2., 1000000) 


In [ ]:
# 2) Von Neumann's rejection method
# If f(x) has positive and negative values on [a,b] then:
W0 =1.0 # greater than or equal to the maximum of abs(f(x)) on [a, b]
f = lambda x: np.sin(x)


def Von_Neumann(f,W0,a,b,N):
    Nmax = 0
    Nmenos = 0
    for i in range(N):
       x = np.random.uniform(a,b)
       y = np.random.uniform(-W0,W0)
       # point inside the positive area of the integral
       if (y > 0)and(y < f(x)): Nmax   += 1 
       # point inside the negative area of the integral
       if (y < 0)and(y > f(x)): Nmenos += 1
       
    # the integral is the positive area minus the negative area
    #  and the area of the rectangle is 2W0(b-a).
    I = (Nmax - Nmenos)/N*(2*W0*(b-a))
                                       
    return I

# Example3:
W0 =1.0 # greater than sin(x) on [0,pi/2] 
f = lambda x: np.cos(x) 
x = np.linspace(0,np.pi,1000)
plt.fill_between(x,f(x))

Von_Neumann(f,W0,0.0, np.pi,1000000) 


<a id='Integrales_múltiples'></a>
# 8) Multiple integrals 
Suppose you want to compute the electronic properties of a silicon atom, which has $14$ electrons; then for each electron the dimension increases by three (coordinates $(x,y,z)$), and the integral will be of dimension $3\times14=42$, meaning you need to solve an integral of the form,

$$\underbrace{\int dx_1...\int dx_{42}}_{42 \text{ integrals}} \,f(x_1,...,x_{42})
\approx \underbrace{\sum_{i_1=1}^n\, ... \sum_{i_{42}=1}^n }_{42 \text{ sums}}w_{i_1 ,..., i_{42}}f(x_{i_1},...,x_{i_{42}}),
$$

where each integral is approximated by a sum up to $n$ (i.e. $42$ nested `for` loops, each from $1$ to $n$); now, if the Gauss method is used with $n=6$, you need to compute $n^{42}=6^{42}=10^{32}$ integration points; on a fast computer, suppose $10^6$ points are computed per second - the calculation would then take $10^{26}$ s, which is more than the age of the universe, of order $\sim10^{17}$ s (task: repeat this same calculation for the carbon atom, which has $6$ electrons, and use Simpson with $n=100$). In conclusion, traditional methods don't work for multidimensional integrals, so these integrals have to be solved with statistical techniques; let's see.


From the mean value theorem we know the value of the integral can be computed as

$$A=\int_a^bf(x)\,dx= (b-a)\bar{f}\approx\frac{b-a}{N}\sum_{i=1}^Nf(x_i),$$

this tells us that we can generate $N$ values $x_i$ randomly on the interval $[a,b]$, evaluate the function at each of them, and thus compute the integral by computing the average $\bar f$; this defines a Monte Carlo method for solving integrals and it can be generalised to several dimensions,

$$I = \int_{a_1}^{b_1}\cdots\int_{a_n}^{b_n}f({\bf x})\,d^n{\bf x}\approx\frac{(b_1-a_1)(b_2-a_2)\cdots(b_n-a_n)}{N}\sum_{i=1}^Nf({\bf x}_i),$$

in this case $N$ vectors ${\bf x}_i=\{x_1^i,x_2^i,\cdots x_n^i\}$ are generated with components with random values inside the $n$-dimensional volume, and the vectors ${\bf a}$, ${\bf b}$ define the limits of the integral.

It can be shown that the error for a normal distribution decreases with the number of points as $\frac{1}{\sqrt{N}}$, more precisely as, 
 
$$
\begin{align}
\sigma_A=&\frac{1}{\sqrt{N}}\sigma_f,\\
\sigma_A=&\frac{1}{\sqrt{N}}\sqrt{\frac{1}{N}\sum_{i=1}^N (f({\bf x}_i)-\bar{f})^2},
\end{align}
$$ 

where $N$ points are distributed over $n$ dimensions, as opposed to, for example, Simpson, where $N/n$ points are used for each integration. It can be shown that if $N$ is fixed, the error increases with $n$; moreover, the error will be proportional to $N$ times the error in each integral; in general, for large $n$ the error will be smaller in the Monte Carlo calculation, and even for $n$ equal to $3$ or $4$ the Monte Carlo error is already similar to the error of conventional methods.

**Example**:
Use Monte Carlo to find the value of the integral,

$$I = \int_0^1dx_1\int_0^1dx_2 \cdots\int_0^1dx_n  \sqrt{(x_1+x_2+\cdots+x_n)}$$

Solution: for $n=100$ we have:


In [ ]:
import numpy as np

N   = 100000 # number of vectors computed
dim = 100    # dimension n of the integral
I = 0
for i in range(N): # compute average of f(x)
    x = np.random.rand(dim) 
    I = I + np.sqrt(sum(x))
    
I/N # average


**Exercise**: Compute the error $\sigma_A$ of the previous integral.


**Task**: Solve the following integrals with the proposed methods and compute the error for,

$$
\begin{eqnarray}
a)&&\int_0^1 \frac{x^4(1-x)^4}{1+x^2}\;\mathrm{d}x=\frac{22}{7}-\pi.\\
b)&&\int_0^\infty \frac{x^3}{e^x-1}\;\mathrm{d}x=\frac{\pi^4}{15}\quad\hbox{(appears in the Debye theory for heat capacity in crystals).}\\
c)&&\int_0^1 x^{-x}\;\mathrm{d}x=\sum_{n=1}^\infty n^{-n}\quad\hbox{(known as the "Sophomore's dream").}\\
d)&&\int_0^1 [\ln(1/x)]^p\;\mathrm{d}x=!p \quad\hbox{ if } \quad 0 \le p \le 10.\\
e)&&\int_0^{2\pi} e^{z\cos\theta}\;\mathrm{d}\theta=2\pi I_0(z)\quad\hbox{(where}\, I_0(z)\,\hbox{is the Bessel function of the first kind defined on}\, 0 \le z \le 2).\\
f)&&\int _{0}^{\infty }\cos t^{2}\,dt=\int _{0}^{\infty }\sin t^{2}\,dt={\sqrt {\frac {\pi }{8}}}.
\end{eqnarray}
$$
Compare the results of the algorithms given in this chapter against those given by `scipy.integrate.quad(f,a,b)` (note this routine lets you use infinity as `np.inf`).

**Task:** The Fresnel integrals are defined by the following expansions, which can also be expressed as power series that converge for every $x$:

$$
\begin{aligned}
S(x)&=\int _{0}^{x}\sin(t^{2})\,dt=\sum _{n=0}^{\infty }(-1)^{n}{\frac {x^{4n+3}}{(4n+3)(2n+1)!}}\\
C(x)&=\int _{0}^{x}\cos(t^{2})\,dt=\sum _{n=0}^{\infty }(-1)^{n}{\frac {x^{4n+1}}{(4n+1)(2n)!}}
\end{aligned} 
$$

a) Plot $S(x)$ and $C(x)$ for $0\leq x \leq 10$.

b) The Cornu spiral, also known as the clothoid (created by Marie Alfred Cornu as a nomogram for optical diffraction calculations, and also useful as a transition curve when laying out highways or railways, since a vehicle following this curve at constant speed will have constant angular acceleration), is the curve whose parametric equations are given by,

$$
C'(t)^{2}+S'(t)^{2}=\sin ^{2}(t^{2})+\cos ^{2}(t^{2})=1
$$

Plot the spiral in the plane $(-1 \leq x,y \leq 1)$. 

**Task**: Consider the function $f(\mathbf{x})=|\mathbf{x}|$; compute its integral on the interval $0\le x_i\le 1$ for dimension $100$, compute the error as a function of $N$ and plot it to verify that it decays as $\frac{1}{\sqrt{N}}$, use `randn()` (normal distribution) instead of `random()` (uniform distribution). Hint: use the dot product.

**Task**: It's easy to show by induction that the following multidimensional integral has an analytical solution,

$$I(n) = \int_{-1}^1dx_1\int_{-1}^1dx_2 \cdots\int_{-1}^1dx_n {(x_1^2+x_2^2+\cdots+x_n^2)} = \frac{n2^n}{3}.$$

a) Verify the answer by Monte Carlo and compute the error for $n=100$. b) plot the error as a function of $n$ and verify that it decays as $\frac{1}{\sqrt{N}}$.

**Task**: integrate the Gaussian function,

$$f(\mathbf{x}) =  \frac{1}{\sqrt{(2\pi)^D}}\exp\left(-\frac{1}{2}(\mathbf{x}-10\mathbf{\hat u})^2 \right)$$

on the interval $7\le x_i\le 13$ for dimension $D=100$ and compute the error, where $\mathbf{\hat u}=\mathbf{\mathbf{x}}/|\mathbf{x}|$ .

**Task**: In electromagnetism it can be shown that the potential produced by a ring of radius $a=1.0$ m in the $xy$ plane with charge distribution $\lambda$, at a point P off the $z$ axis, is given by,

$$V = \frac{\lambda}{2 \pi \varepsilon_0}\sqrt{\frac{am}{x}} K(m)$$

where, 

$$m = \frac{2}{1 + \frac{z^{2} + x^{2} + a^{2}}{2 a x}},$$ 

and $K(m)$ is the complete elliptic integral of the first kind defined as,

$$K(m) = \int_{0}^{\pi/2} \frac{d  \phi}{\sqrt{1 - m \sin^{2}(\phi)}}$$

a) If $\frac{\lambda}{2\pi\varepsilon_0}=1.0$, what is the electric potential at the point $(x,y,z)=(1.5,0,1.0)$?<br>
b) Plot the solution for $-100<x<100$, using steps of $0.01$. What can you conclude from the plot?<br> 
c) The analytical solution is given by the series,

$$ K(m)={\frac {\pi }{2}}\sum _{n=0}^{\infty }\left({\frac {(2n)!}{2^{2n}(n!)^{2}}}\right)^{2}m^{n}={\frac {\pi }{2}}\sum _{n=0}^{\infty }{\bigl (}P_{2n}(0){\bigr )}^{2}m^{n},$$

where $P_n$ are the [Legendre polynomials](#MATERIAL_COMPLEMENTARIO). Equivalently, also,

$$K(m)={\frac {\pi }{2}}\left(1+\left({\frac {1}{2}}\right)^{2}m+\left({\frac {1\cdot 3}{2\cdot 4}}\right)^{2}m^{2}+\cdots +\left({\frac {\left(2n-1\right)!!}{\left(2n\right)!!}}\right)^{2}m^{n}+\cdots \right),$$

where $n!!$ is the double factorial. Plot and compare these three solutions against the one given by the numerical integral. What do you conclude? (Note: be careful with $x\leq 0$ in the series; consider symmetry for this).

**Task**: From the potential of the previous problem, compute the electric field at P if,

$$\mathbf{E}=\left(-\frac{dV}{dx},-\frac{dV}{dy},-\frac{dV}{dz} \right).$$

(Use the centred numerical derivative) How does the electric field vary on $-1.5<x<1.5$?

**Task**: Gauss-Chebyshev quadratures are defined as,

$$
\int _{-1}^{1}{\frac {f(x)}{\sqrt {1-x^{2}}}}\,dx
\quad\hbox{ and }\quad
\int _{-1}^{1}{\sqrt {1-x^{2}}}g(x)\,dx.$$

Both are solved by Gaussian quadrature as,

$$
\int _{-1}^{1}{\frac {f(x)}{\sqrt {1-x^{2}}}}\,dx\approx \sum _{i=1}^{n}w_{i}f(x_{i})
$$

where for the first integral,

$$
x_{i}=\cos \left({\frac {2i-1}{2n}}\pi \right)
\quad\hbox{ and }\quad
w_{i}={\frac {\pi }{n}},
$$

and for the second,

$$
x_{i}=\cos \left({\frac {i}{n+1}}\pi \right)
\quad\hbox{ and }\quad
w_{i}={\frac {\pi }{n+1}}\sin ^{2}\left({\frac {i}{n+1}}\pi \right).\,
$$

Implement these in python.

**Task**: In Maxwellian relativity (distribution $f(v) \propto e^{-\gamma\frac{mc^2}{K_BT}}$, with $\gamma=\frac{1}{\sqrt{1-\beta^{2}}}$ and $\beta= v/c$), the average energy of an electron is computed as,

$$
\langle E\rangle = \frac{mc^2}{\lambda/c} \int _{-1}^{1} \gamma f(\beta c)\,d\beta,
$$

(where $\lambda$ is the normalisation of the integral $\lambda = \int _{-c}^{c}f(v)\,dv$), which results in the integral,

$$
I = \int _{-1}^{1} \frac{1}{\sqrt {1-x^{2}}}e^{-\frac {1}{\sqrt{1-x^{2}}}\frac{mc^2}{K_BT}}\,dx
$$

and $\lambda/c$ can be manipulated as,

$$
\frac{\lambda}{c} = \int _{-1}^{1} \frac{\sqrt {1-x^{2}}}{\sqrt {1-x^{2}}}e^{-\frac {1}{\sqrt{1-x^{2}}}\frac{mc^2}{K_BT}}\,dx
$$

to be computed with Gauss-Chebyshev quadrature.
Compute the value of $\langle E\rangle$ for $m=10^{ -27}$ g, $c=3\times10^{10}$ cm/s, and $T=5\times10^{8}$ K, $K_B=1.38064852\times10^{-16}$ erg/K, use $n=3$ and $n=4$.

**Task**: A body of mass $m$ travels vertically from the Earth's surface; if air resistance is neglected but gravity is included, the escape velocity is given by,

$$v^2=2gR\int_1^\infty z^{-2}\,dz\quad\hbox{where}\quad z=\frac{x}{R},$$

where $R=3960$ miles is the Earth's radius and $g=0.00609$ miles/s$^2$ the gravitational constant; compute $v$.


<a id='cuadratura_problemas'></a>
**Task**: Modify the Gauss-Legendre routine and implement:<br>
a) *Gauss-Laguerre quadrature*,

$$
\int _{0}^{\infty }e^{-x}f(x)\,dx\approx \sum _{i=1}^{n}w_{i}f(x_{i})
\quad\hbox{ with }\quad
w_{i}={\frac{x_{i}}{(n+1)^{2}[L_{n+1}(x_{i})]^{2}}},
$$

where the Laguerre polynomials and their derivatives are generated from the relations,

$$
\begin{align}
L_0(x)&=1,\\
L_1(x)&=-x+1,\\
L_{k + 1}(x) &= \frac{(2k + 1 - x)L_k(x) - k L_{k - 1}(x)}{k + 1},\\
L'_k(x) &= -L_{k-1}(x).
\end{align}
$$

b) *Gauss-Hermite quadrature*,

$$
\int _{-\infty }^{\infty }e^{-x^{2}}f(x)\,dx\approx \sum _{i=1}^{n}w_{i}f(x_{i})
\quad\hbox{ with }\quad
w_{i}={\frac {2^{n-1}n!{\sqrt {\pi }}}{n^{2}[H_{n-1}(x_{i})]^{2}}},
$$

where the Hermite polynomials and their derivatives are generated from the relations,

$$
\begin{align}
H_0(x)&=1,\\ 
H_1(x)&=2x,\\
H_{n+1}(x)&=2xH_{n}(x)-2nH_{n-1}(x),\\
H_{n}'(x)&=2nH_{n-1}(x).
\end{align}
$$

c) Verify your implementations by comparing against the integrals (gamma function and Gaussian bell),

$$
\int _{0}^{\infty }{\sqrt {x}}\,e^{-x}\,dx={\frac {1}{2}}{\sqrt {\pi }}
\quad\hbox{ and }\quad
\int _{-\infty }^{\infty }e^{-x^{2}}\,dx={\sqrt {\pi }}.$$

Plot the relative errors as a function of $n$.

**Task:** Use the Laguerre method to plot the following function 

$$
I(x)=\int_0^{\infty}  x^3 \exp(-x^2 t) \, \mathrm{d}t = x
$$

for $-50\leq x\leq 50$ (use the routine given in the supplement to obtain the zeros and weights). Note that the error will dominate the plot (it isn't equal to $x$); make the change of variable $u=x^2t$ and re-plot - what do you conclude? Compute the integral with Legendre quadrature and compare.

**Task**: Write a python routine that generates the first ten polynomials and their derivatives for a) Hermite b) Laguerre (see the [supplement](#Polinomios_de_legendre)).

**Task**: The mean value of a function $f$ with Gaussian noise is computed as,

$$
\langle f\rangle  =\int _{-\infty }^{\infty }{\frac {1}{\sigma {\sqrt {2\pi }}}}\exp \left(-{\frac {(y-\mu )^{2}}{2\sigma ^{2}}}\right)f(y)dy
$$

If the substitution is made,

$$
 x={\frac {y-\mu }{{\sqrt {2}}\sigma }}\Leftrightarrow y={\sqrt {2}}\sigma x+\mu
$$

a) show that this gives the modified Gauss-Hermite quadrature,

$$
\langle f\rangle =\int _{-\infty }^{\infty }{\frac {1}{\sqrt {\pi }}}\exp(-x^{2})f({\sqrt {2}}\sigma x+\mu )dx
$$

i.e.,

$$
\langle f\rangle \approx {\frac {1}{\sqrt {\pi }}}\sum _{i=1}^{n}w_{i}h({\sqrt {2}}\sigma x_{i}+\mu ). 
$$

b) Compute the mean value of $f(x)=\cos(x)$.

See more [physics examples](http://www.sc.ehu.es/sbweb/fisica3/especial/eliptica/eliptica.html) with elliptic integrals. 

**Task**: [Heat capacity](https://en.wikipedia.org/wiki/Heat_capacity) or thermal capacity of a body, $C_V$, is defined as the ratio between the amount of heat energy transferred to a body and the change in temperature it undergoes, i.e. $C_V$ measures the energy needed to raise the temperature of a body by one unit of temperature. According to Debye theory, the heat capacity is given by the integral,

$$
\frac {C_{V}}{Nk}=9\left({T \over T_{\rm {D}}}\right)^{3}\int _{0}^{T_{\rm {D}}/T}{x^{4}e^{x} \over \left(e^{x}-1\right)^{2}}\,dx,
$$

where $T_D$ is the Debye temperature, $k$ is Boltzmann's constant and $N$ the number of particles contained in the volume $V$. a) Use the integration methods to plot the heat capacity, $C_V/3Nk$, as a function of $T/T_D$ on the interval $[0,1.5]$.
b) Einstein had initially proposed the model,

$$\frac{C_V}{3Nk}=\left({\epsilon\over k T}\right)^2{e^{\epsilon/kT}\over \left(e^{\epsilon/kT}-1\right)^2},$$

with energy given by $\frac{\epsilon}{kT_D} =\frac{\hbar \omega}{kT_D} = \sqrt[{3}]\frac{\pi}{6}=0.805995977$, where $\omega$ is defined as the oscillator frequency used in Einstein's model. 
Plot and compare against the Debye model (Note: the models are similar but Einstein's fails at low temperatures).


<a id='MATERIAL_COMPLEMENTARIO1'></a>
# Supplementary material

## Proof of the trapezoidal and Simpson rules using Lagrange polynomials
From Lagrange's theorem we know that every function can be represented by a degree-$n$ polynomial for $n+1$ given points plus the error, i.e.,

$$f(x) = P_n(x) + \epsilon(x).$$

In the *trapezoidal* method the function is approximated by a degree-one polynomial
on the interval $[x_0,x_1]$, and integrated, 

$$
\begin{align} 
 \int_{x_0}^{x_1}f(x)dx =
 \int_{x_0}^{x_1}\left(\frac{(x-x_1)}{(x_0-x_1)}f(x_0) 
 + \frac{(x-x_0)}{(x_1-x_0)}f(x_1)\right)dx
 + \int_{x_0}^{x_1}\left(\frac{1}{2}(x-x_0)(x-x_1)f^{''}(\xi)\right)dx. 
\end{align}
$$

Which gives the result,

$$\int_{x_0}^{x_1}f(x)dx = \frac{h}{2}[ f(x_0) + f(x_1) ]-\frac{h^3}{12}f^{''}(\xi).$$

For *Simpson's* method, three points are considered on the interval $[x_0,x_2]$ with $x_1$ as the midpoint; this gives a degree-two polynomial plus the error, which when integrated,

\begin{align}
\int_{x_0}^{x_2}f(x)dx =& \int_{x_0}^{x_2}\left(\frac{(x-x_1)(x-x_2)}{(x_0-x_1)(x_0-x_2)}f(x_0) + \frac{(x-x_0)(x-x_2)}{(x_1-x_0)(x_1-x_2)}f(x_1) 
+ \frac{(x-x_0)(x-x_1)}{(x_2-x_0)(x_2-x_1)}f(x_2) \right)dx \\
+& \int_{x_0}^{x_2}\left(\frac{1}{6}(x-x_0)(x-x_1)(x-x_2)f^{(3)}(\xi)\right)dx
\end{align}

but gives a result of $O\left(h^4\right)$ involving $f'''(\xi)$; a much better alternative method is to use a Taylor series up to degree 4 and integrate,

$$
\int^{x_2}_{x_0} f(x)dx =\left[ f(x_1)(x - x_1) + \frac{f'(x_1)}{2}(x - x_1)^2 
+ \frac{f''(x_1)}{6}(x - x_1)^3 + \frac{f'''(x_1)}{24}(x - x_1)^4\right]^{x_2}_{x_0}
+ \frac{1}{24}\int^{x_2}_{x_0}f^{(4)}(\xi(x))(x - x_1)4 dx,
$$

if we define $h = x_2 - x_1 = x_1 - x_0,$ this gives the result,

$$\int_{x_0}^{x_2} f(x)dx = \frac{h}{3}[ f(x_0)+4f(x_1)+f(x_2) ]-\frac{h^5}{90}f^{(4)}(\xi).$$

**Task**: fill in the intermediate steps in the proofs.


<a id='Deducción_del_método_de_Romberg'></a>
## Derivation of Romberg's method
Consider the integral
 $$I=\int_a^bf(x)dx$$
 has an approximate value $A(h)$ computed from the trapezoidal method, so if the error is included as a series of even powers of $h$,
 
 $$I = A(h) + K_1h^2 +K_2h^4 + K_3h^6...$$
 
 Suppose that, to reduce the error, $A$ is computed but at each $h/2$,
 
 $$I = A\left(\frac{h}{2}\right) + K_1\frac{h^2}{4} +K_2\frac{h^4}{16} ...$$
 
 If this equation is multiplied by $4$ and the first is subtracted from it (thereby eliminating the $h^4$ term), we obtain
 
 $$I =\left[\frac{4}{3}A\left(\frac{h}{2}\right)-\frac{1}{3}A\left(h\right)\right] + K_2\frac{h^4}{4} ...$$

the expression between square brackets has an error of order $O(h^4)$ and is therefore closer to the true value $I$; so, if $A(h)$ is redefined as $A_1(h)$ and the term between brackets as $A_2(h)$, and the process is repeated $j$ times, we obtain the recursive formula,

$$A_j(h)=A_{j-1}\left(\frac{h}{2}\right)+
\frac{A_{j-1}\left(\frac{h}{2}\right) - A_{j-1}(h)}{4^{j-1}-1}.$$

with error of order $O(h^{2j})$, so

$$I=A_j(h)+O\left(h^{2j}\right).$$

Since each new $A_j$ is closer to $I$, the algorithm ends when the relative error is smaller than a given $\epsilon$, i.e.

$$\left|\frac{A_j(h)-A_{j-1}(h)}{A_{j}(h)}\right|<\epsilon.$$

In Romberg's method, the first step ($j=1$) starts with a single trapezoid, i.e. $n=1$ and $h=(b-a)$, so

$$A_1(x)=(b-a)\left(\frac{f(a)+f(b)}{2}\right)$$

for the second step ($j=2$), $h/2=(b-a)/2$ is divided, so $n=2$ (note that $A(h/2)$ is evaluated with $2$ trapezoids, one on the interval $[a,a+h/2]$ and the other on $[a+h/2,b]$), so

\begin{align}
A_2(h)=& \frac{1}{3}\left[4A\left(\frac{h}{2}\right)-A\left(h\right)\right]\\
      =& \frac{1}{3}\left[4\frac{(b-a)}{2}\left(\frac{f(a)+f(a+h/2)}{2}\right)
                         +4\frac{(b-a)}{2}\left(\frac{f(a+h/2)+f(b)}{2}\right)
                         -(b-a)\left(\frac{f(a)+f(b)}{2}\right)
                   \right]\\
      =& \frac{(h/2)}{3}\left[f(a)+4f(a+h/2)+f(b)\right],             
\end{align}

which is equivalent to Simpson's method on the points $[a,b]$ with $a+h/2$ as the midpoint.

If we switch to a more convenient two-index notation, with one index $j$ that increases the extrapolation step and another index $k$ that, at each extrapolation step, counts how many trapezoids to sum, Romberg's method can be redefined recursively as follows (see Burden page 216 or [wikipedia](https://en.wikipedia.org/wiki/Romberg%27s_method)):

$$ h_k = \frac{b-a}{2^k},\quad \hbox{ and }\quad n_k=\frac{(b-a)}{h_k},$$

which implies $n_k=2^k$. The first element is defined by

$$R_{0,0} \overset{\mathrm{def}}{=} A_1(x)=\frac{1}{2} (b-a) (f(a) + f(b)),$$

then for $k>0$ we have the trapezoidal approximation

$$R_{k,0} = \frac{1}{2} R_{k-1,0} + h_k \sum_{k=1}^{2^{k-1}} f(a + (2k-1)h_k)\quad\hbox{ for } k=1,2,...$$

In general (compare against $A_j(h/2)$, but with $j$ starting at zero instead of one)

$$R_{k,j} = R_{k,j-1} + \frac{1}{4^j-1} (R_{k,j-1} - R_{k-1,j-1})\quad\hbox{ for } k=j,j+1,...$$

which is equivalent to

$$R_{k,j} = \frac{1}{4^j-1} ( 4^j R_{k,j-1} - R_{k-1,j-1})\quad\hbox{ for } k=j,j+1,...$$

which generates the following table

$$
\begin{array}{l}
\hline &\hline    &\hline  &\hline &\hline \\[-2pt]
 k && O(h_k^2)   && O(h_k^4)   && O(h_k^6)  && O(h_k^8)     && O(h_k^{2k_{max}})\\
 \hline &\hline    &\hline  &\hline &\hline \\
 1 && R_{0,0}    &&            &&           &&              &&          \\
 2 && R_{1,0}    && R_{1,1}    &&           &&              &&          \\
 3 && R_{2,0}    && R_{2,1}    && R_{2,2}   &&              &&          \\
 4 && R_{3,0}    && R_{3,1}    && R_{3,2}   && R_{3,3}      &&          \\
 \vdots&& \vdots &&\vdots      &&  \vdots   &&              &\ddots&    \\
 k_{max} && R_{k_{max},0}    && R_{k_{max},1}    && R_{k_{max},2}   && R_{k_{max},3}      &\cdots&R_{k_{max},k_{max}}\\
\hline &\hline&\hline  &\hline &\hline
\end{array}
$$

The asymptotic upper bound on the error of $R_{k,j}$ is:

$$O\left(h_k^{2^{j+1}}\right). $$

As before, the zeroth-order extrapolation $R_{k,0}$ is equivalent to the trapezoidal rule with $2^k+1$ points, and the first-order extrapolation $R_{k,1}$ is equivalent to Simpson's rule with $2^k+1$ points.


<a id='Polinomios_de_legendre'></a>
## Legendre polynomials
The [Legendre polynomials](https://en.wikipedia.org/wiki/Legendre_polynomials#Additional_properties_of_Legendre_polynomials) can be built from the recurrence relation,

$$(n+1) P_{n+1}(x) = (2n+1) x P_n(x) - n P_{n-1}(x)\,$$

and their derivatives can be computed from 

$$ {d \over dx} P_n(x) = \frac{n}{x^2-1}\left(xP_n(x) - P_{n-1}(x) \right).$$

The following code generates the first 5 Legendre polynomials $P_n(x)$ starting from $P_0=1$ and $P_1=x$, needed for the Gauss-Legendre integral (not to be confused with the Lagrange polynomials):


In [ ]:
from sympy import *
from IPython.display import display # to print in latex
init_printing() #init_printing(use_latex='mathjax')
#init_session() 

x = Symbol('x')
t = np.linspace(-1,1,1000)
plt.plot(t, t, label="P_1")

p0=1; p1=x            # P_0 = 1 and P_1 = x
display(p1)           # print(p1) also works
for n in range(1,5):  # the rest of the P_n are obtained recursively:
   p2 = ((2*n+1)*x*p1-n*p0)/(n+1) # (n+1)P_(n+1) = (2n+1)xP_n - nP_(n-1)
   p0 = p1; p1 = p2
   # recurrence relation for the derivative of P_n(x)
   # pp = (x*p1 - p0)*n/(x**2. -1)  # P'_n(x) = n(xP_n - P_(n-1))/(x^2-1)
   
   display(simplify(p2)) # print the polynomial
   f = lambdify(x, p2)   # plot the polynomial
   plt.plot(t, f(t), label="P_%d"%(n+1))

plt.legend()
plt.grid()
plt.show()


In [ ]:
# another way to express the recurrence relation:
#-------------------------------------------------------------
p1 = 1 ; p2 = 0          # find the ith root.
for n in range(1, 6):
   p3 = p2               # Recurrence relation of 
   p2 = p1               # P_n(x).
   p1 = ( (2*n - 1)*x*p2 - (n - 1)*p3)/n
   display(simplify(p1)) # print the polynomial
   f = lambdify(x, p1)   # plot the polynomial

   plt.plot(t, f(t), label="P_%d"%n)

plt.legend()
plt.grid()
plt.show()


## Laguerre and Hermite polynomials
### Zeros and weights of the Laguerre and Hermite functions with Sympy
The following routine finds the zeros $x_i$ and weights $w_i=w(x_i)$ of the degree-$n$ [Laguerre and Hermite polynomials](https://halshs.archives-ouvertes.fr/halshs-01843715/document) (and Legendre), needed for Gaussian integration using `Sympy`:


In [ ]:
import sympy as sp

def gauss_cuadratura(polinomio, n, digits = 17):
    x = sp.symbols('x')                             # variable
    raices = sp.Poly(polinomio(n, x)).all_roots()   # get analytical roots
    
    xi = [raiz.evalf(digits) for raiz in raices]    # get numerical roots and weights
    wi = [(raiz/((n+1)*polinomio(n+1, raiz))**2).evalf(digits) for raiz in raices]
    
    return xi, wi

gauss_cuadratura(sp.laguerre, 4)


In [ ]:
gauss_cuadratura(sp.hermite, 4)


In [ ]:
gauss_cuadratura(sp.legendre, 6, 10)


## Riemann, trapezoidal and Simpson methods via Lagrange interpolation


In [ ]:
# Plot of the integration areas for the
# Riemann, Trapezoidal and Simpson methods. 
# To understand how these methods work, try
# different values of nmax and other functions.
#-------------------------------------------------------------
import numpy as np
import matplotlib.pylab as plt
import scipy.interpolate

def area_curva(f,a,b,nmax=8,n=2):
    D = (b-a)/nmax                                   # delta x.
    N = nmax
    if n == 2:                                       # If Simpson, then
        N = nmax//2                                  # halve nmax and
        D = D*2                                      # multiply the delta by 2.

    x = np.linspace(a,b,1000)                        # split x into a thousand points and 
    plt.plot(x,f(x),c='b',lw=2)                      # plot the function on [a,b].
    plt.plot(a,f(a),'o',c='k')                       # plot the first point.
    plt.plot([b,b],[0,f(b)],'--',c='r',lw=1)         # plot vertical line.

    for i in range(N):                               # sum over areas A1,A2,...,An.
        xi = a + i*D                                 # integration points.
        x  = np.linspace(xi,xi+D)                    # values of x on [xi,xi+D].
        
        if   n == 0:                                 # degree-0 polynomial (Riemann),
            xj = np.array([xi+D])                    # a single interpolation point.
            pn = scipy.interpolate.lagrange(xj,f(xj))
            
        elif n == 1:                                 # degree-1 polynomial (Trapezoidal),
            xj = np.array([xi,xi+D])                 # two interpolation points.
            pn = scipy.interpolate.lagrange(xj,f(xj)) 
  
        elif n == 2:                                 # degree-2 polynomial (Simpson),
            xj = np.array([xi,xi+D/2,xi+D])          # three interpolation points.
            pn = scipy.interpolate.lagrange(xj,f(xj))
            plt.plot(xi+D/2,f(xi+D/2),'o',color='k') # plot points.
            
        Alpha = .5 if i == N//2 else .1              # transparency of the plotted area.
        plt.fill_between(x,pn(x),color="r",lw=2,alpha=Alpha) # Plot the red area.
        plt.plot(xi+D,f(xi+D),'o',color='k')         # plot points.
        plt.plot([xi,xi],[0,f(xi)],'--',c='r',lw=1)  # plot vertical lines.
        
        if (nmax!=8):                                # show area number.
            plt.text(xi+.1*D, 2, r'$A_{%d}$'%(i+1))   

            
    if nmax == 8: # Change x,y axes for the general [x_0, ..., x_n] plot.   
        x_labels = [r'$x_0=a$',r'$x_1$',r'$x_2$','...', r'$x_{i-1}$',
                    r'$x_i$',r'$x_{i+1}$ ...',r'$x_{n-1}$',r'$x_{n}=b$']
        xv = np.linspace(a,b,len(x_labels))
        
        plt.yticks([0,5],['0','f(x)'],fontsize=14,)  # change y-axis values.
        plt.xticks(xv, x_labels, fontsize=14)        # change x-axis values.
        plt.text((nmax+D)/2, 2, r'$A_i$',size=18)    
        plt.gca().spines['right'].set_color('none')  # remove the right-hand line.
        plt.gca().spines['top'].set_color('none')    # remove the top line.


In [ ]:
# Run this cell to see the notebook's plots 

nmax = 8   # number of areas to sum (must be even for Simpson).
a,b  = 0,8                                           # interval.                 
f = lambda x: np.sin(x*2.2)+np.exp(0.1*x*2)+2        # function to integrate.

plt.figure(1,figsize = (8,5))
area_curva(f,a,b,nmax,0) # Riemann
plt.savefig("../figures/Riemann.png",transparent=True,format='png')

plt.figure(2,figsize = (8,5))
area_curva(f,a,b,nmax,1) # Trapezoidal
plt.savefig("../figures/Trapezoidal.png",transparent=True,format='png')

plt.figure(3,figsize = (8,5))
area_curva(f,a,b,nmax,2) # Simpson
plt.savefig("../figures/Simpson.png",transparent=True,format='png')


In [ ]:
# Improper-integral function
a,b = 1,4
x=np.linspace(a+0.02,b,1000)
plt.plot(x,f(x)/(x-1)**.2)
plt.plot([a,a],[0,10],"r--")
plt.plot([b,b],[0,f(b)/(b-1)**.2],"r-")

plt.yticks([0,8],['0','f(x)'],fontsize=14,) # change y-axis values.
plt.xticks([a,b], ['a','b'], fontsize=14)    # change x-axis values.
plt.ylim(0,10)
plt.xlim(0,b+2)
plt.gca().spines['right'].set_color('none')  # remove the right-hand line.
plt.gca().spines['top'].set_color('none')    # remove the top line.
plt.text(1.5, 5, r'$f(x)=\dfrac{g(x)}{(x-a)^p}$',size=14) 
plt.savefig("../figures/f_impropia.png",transparent=True,format='png')


## Bibliography
the proof of Gaussian integration can be found in the book "*Computation in Modern Physics*" by William R. Gibbs, [page 14](https://books.google.com.co/books?id=GyUIqGOVBdQC&pg=PA16&lpg=PA16&dq=Gauss%E2%80%93Laguerre+integral+physics+problem&source=bl&ots=Sakvvm4XsQ&sig=ACfU3U3EUGpqfPrNvlm4UqOMl2wGk8cHgg&hl=es-419&sa=X&ved=2ahUKEwiQ_4uaxf3iAhVCs1kKHZjfDqgQ6AEwBnoECFwQAQ#v=onepage&q=Gauss%E2%80%93Laguerre%20integral%20physics%20problem&f=false), $3^{th}$ edition.

[Oscillatory integrals with Gauss-Laguerre](https://www.sciencedirect.com/science/article/pii/S0377042713003385).

https://physics.bgu.ac.il/COURSES/IntroCompPhys/2019B/HW/

https://www.researchgate.net/publication/260799153_Numerical_integration_over_n-dimensional_cubes_using_generalized_Gaussian_quadrature
